# Taxi Route Revenue, Pricing & Fleet Planner

This notebook analyzes three Kampala taxi routes using ten days of passenger counts. The reusable route, forecaster, equilibrium and demand-generator classes are in [`src/taxi.py`](src/taxi.py). The non-Ntinda route data and the 60-day extension are illustrative.

## Assumptions and units

Passenger counts are assumed to be total one-way passenger trips per route per day. Gross revenue is passenger count × fare, before operating costs, discounts, or empty seats. Fleet sizing treats each vehicle as carrying up to 14 passengers on each of 8 one-way trips per day, then applies the specified 15% planning buffer. Actual operations may need separate peak-period, direction, maintenance, and load-factor adjustments.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.taxi import (
    ExponentialSmoothingForecaster,
    ForecasterEvaluator,
    LinearTrendForecaster,
    MovingAverageForecaster,
    Route,
    SeasonalNaiveForecaster,
    SupplyDemandEquilibrium,
    WeeklyDemandGenerator,
)

ROUTE_DATA = {
    "Kampala-Ntinda": ([35, 40, 42, 50, 55, 60, 48, 52, 47, 45], 2_000),
    "Kampala-Entebbe": ([60, 58, 65, 70, 72, 80, 75, 68, 66, 64], 5_000),
    "Kampala-Mukono": ([45, 47, 50, 49, 55, 62, 58, 53, 51, 50], 3_000),
}
routes = {
    name: Route(name, passengers, fare)
    for name, (passengers, fare) in ROUTE_DATA.items()
}
DAYS = np.arange(1, 11)

## 1. Route revenues and passenger statistics

Mean and sample variance/standard deviation are calculated from passenger counts using Python's `statistics` module. Variance is in passengers² and standard deviation is in passengers.

In [ ]:
for route in routes.values():
    stats = route.descriptive_statistics()
    print(f"\n{route.name} | fare UGX {route.fare_ugx:,.0f}")
    print(f"Daily revenue: {route.daily_revenue().astype(int).tolist()}")
    print(f"Total revenue: UGX {route.total_revenue():,.0f}")
    print(
        f"Passengers: mean={stats['mean']:.2f}, "
        f"sample variance={stats['variance']:.2f}, "
        f"sample stdev={stats['stdev']:.2f}"
    )

## 2. Ntinda supply–demand equilibrium

Set demand equal to supply: `120 − 0.02P = 10 + 0.03P`. As a linear system in unknowns `P` and `Q`, this becomes `0.02P + Q = 120` and `0.03P − Q = −10`. We solve the 2×2 system with `scipy.linalg.solve`. Compare the implied equilibrium fare with the current UGX 2,000 fare; this simplified model assumes constant linear curves and gives quantity in passengers per trip-hour.

In [ ]:
equilibrium = SupplyDemandEquilibrium()
equilibrium_fare, equilibrium_quantity = equilibrium.solve()
current_ntinda_fare = routes["Kampala-Ntinda"].fare_ugx
print(f"Equilibrium fare: UGX {equilibrium_fare:,.0f}")
print(f"Equilibrium quantity: {equilibrium_quantity:.1f} passengers per trip-hour")
print(f"Current Ntinda fare: UGX {current_ntinda_fare:,.0f}")
position = "below" if current_ntinda_fare < equilibrium_fare else "above"
print(f"The current fare is {position} this modelled equilibrium fare.")
print("This is a model comparison, not a fare recommendation or demand study.")

## 3. Forecasters and walk-forward evaluation

Each origin uses only observations available before the forecasted day. Origins 4–10 correspond to seven one-step-ahead tests. Exponential smoothing's `alpha` is tuned by searching a fixed grid and minimizing this same rolling-origin MAE; that makes the reported tuned score optimistic, so a separate validation period would be needed for unbiased model selection.

In [ ]:
alpha_grid = np.linspace(0.1, 1.0, 19)
route_model_results = {}
for name, route in routes.items():
    best_alpha, alpha_scores = ForecasterEvaluator.tune_alpha(
        route.passenger_counts, alpha_grid
    )
    forecasters = {
        "3-day moving average": MovingAverageForecaster(3),
        f"Exponential smoothing (α={best_alpha:.2f})": ExponentialSmoothingForecaster(best_alpha),
        "Linear trend": LinearTrendForecaster(),
    }
    scores = ForecasterEvaluator.evaluate(route.passenger_counts, forecasters)
    best_name = min(scores, key=scores.get)
    route_model_results[name] = {
        "scores": scores,
        "best_alpha": best_alpha,
        "forecasters": forecasters,
        "best_model": best_name,
    }
    print(f"\n{name} — rolling-origin MAE, passengers/day")
    for model_name, mae in scores.items():
        print(f"  {model_name:38} {mae:.3f}")
    print(f"Best alpha on grid: {best_alpha:.2f}")
    print(f"Best MAE model: {best_name}")

## 4. Forecast day 11 demand and revenue

The day-11 model is selected separately per route by the smallest rolling-origin MAE. Forecast revenue is the passenger forecast multiplied by the route's current fare. The output is a point estimate; these short series do not support a reliable uncertainty interval.

In [ ]:
day_11_forecasts = {}
for name, route in routes.items():
    result = route_model_results[name]
    forecaster = result["forecasters"][result["best_model"]]
    passengers = max(0.0, forecaster.predict_next(route.passenger_counts))
    revenue = passengers * route.fare_ugx
    day_11_forecasts[name] = {
        "passengers": passengers,
        "revenue": revenue,
        "model": result["best_model"],
    }
    print(
        f"{name}: {passengers:.2f} passengers, forecast gross revenue "
        f"UGX {revenue:,.0f} ({result['best_model']})"
    )

### Actual and forecast daily revenue

The line shows observed daily gross revenue. The marker at day 11 is the selected forecaster's estimated revenue.

In [ ]:
figure, axes = plt.subplots(len(routes), 1, figsize=(11, 9), sharex=True)
for axis, (name, route) in zip(axes, routes.items()):
    axis.plot(DAYS, route.daily_revenue() / 1_000_000, "o-", label="Actual", color="tab:blue")
    forecast = day_11_forecasts[name]
    axis.scatter(
        [11], [forecast["revenue"] / 1_000_000], marker="X", s=90,
        label=f"Day 11 forecast ({forecast['model']})", color="tab:orange"
    )
    axis.set_title(name)
    axis.set_ylabel("Gross revenue (UGX millions)")
    axis.grid(alpha=0.25)
    axis.legend()
axes[-1].set_xlabel("Day")
axes[-1].set_xticks(np.arange(1, 12))
figure.tight_layout()
plt.show()

## 5. Fleet planning with a 15% buffer

One vehicle can carry `14 × 8 = 112` passenger trips per day at full capacity under the stated assumption. Fleet size is `ceil(forecast × 1.15 / 112)`, so a fractional vehicle always rounds upward. This is a capacity plan based on total route demand and does not distinguish direction or peak-hour clustering.

In [ ]:
print(f"{'Route':22} {'Day 11 passengers':>20} {'Buffered demand':>18} {'Vehicles':>10}")
for name, route in routes.items():
    passengers = day_11_forecasts[name]["passengers"]
    buffered = passengers * 1.15
    vehicles = route.vehicles_required(passengers, buffer_fraction=0.15)
    print(f"{name:22} {passengers:20.2f} {buffered:18.2f} {vehicles:10d}")

## Extension: seasonal-naïve forecast on 60 synthetic days

Generate reproducible daily demand with a weekly pattern: Fridays (weekday index 4) are busier, Sundays (index 6) are quieter, and small random noise is added. Compare seven-day seasonal naïve forecasts against a three-day moving average using one-step rolling-origin forecasts from day 8 onward. Seasonal naïve repeats the count from the same weekday one week earlier.

In [ ]:
synthetic_demand = WeeklyDemandGenerator.generate(
    base_demand=50,
    days=60,
    seed=20261010,
    noise_std=2.5,
    friday_multiplier=1.3,
    sunday_multiplier=0.7,
)
synthetic_days = np.arange(1, len(synthetic_demand) + 1)
friday_mean = float(np.mean(synthetic_demand[4::7]))
sunday_mean = float(np.mean(synthetic_demand[6::7]))
moving_mae = ForecasterEvaluator.rolling_origin_mae(
    MovingAverageForecaster(3), synthetic_demand, start_index=7
)
seasonal_mae = ForecasterEvaluator.rolling_origin_mae(
    SeasonalNaiveForecaster(7), synthetic_demand, start_index=7
)
print(f"60-day demand: Friday average={friday_mean:.2f}; Sunday average={sunday_mean:.2f}")
print(f"3-day moving average rolling MAE: {moving_mae:.3f}")
print(f"7-day seasonal-naïve rolling MAE: {seasonal_mae:.3f}")
print(f"Seasonal naïve performs better: {seasonal_mae < moving_mae}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(synthetic_days, synthetic_demand, color="slateblue", linewidth=1.5)
plt.scatter(synthetic_days[4::7], synthetic_demand[4::7], color="darkorange", label="Friday")
plt.scatter(synthetic_days[6::7], synthetic_demand[6::7], color="teal", label="Sunday")
plt.xlabel("Day")
plt.ylabel("Passengers/day")
plt.title("Synthetic 60-day taxi demand with weekly pattern")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

The extension uses data generated with a strong recurring weekly pattern, which gives the seasonal-naïve model information the three-day average cannot exploit. Its advantage here is an expected consequence of the synthetic design, not evidence that seasonal naïve will always outperform; operational evaluation should use longer real route data and compare on a reserved test period.

## Conclusions

- Route revenues and sample passenger statistics are calculated directly from the provided 10-day counts and fares.
- Under the assumed Ntinda curves, equilibrium is **UGX 2,200** and **76 passengers per trip-hour**; the current UGX 2,000 fare is below that modelled equilibrium. This does not by itself justify raising fares.
- Rolling-origin evaluation compares moving average, tuned exponential smoothing and linear trend per route. Grid-search MAE is measured on the tuning origins, so the selected score is optimistic.
- Day-11 revenue uses each route's lowest-MAE model. Fleet size includes a 15% buffer and rounds vehicles up, but ignores peak distribution and vehicle availability.
- On the seeded synthetic weekly-pattern data, seasonal naïve should outperform a short moving average because it repeats the same weekday from the previous week; real data may differ.

Use the forecasts as planning aids, and validate them against longer operational data before changing fares or fleet allocation.